In [ ]:
# !ls -lh

from pathlib import Path

for path in sorted(Path.cwd().iterdir()):
    print(path.name)

total 9.8M
-rw------- 1 root root 892K May 19 01:49 niletts_llm_qa_test.csv
-rw------- 1 root root 7.5M May 19 01:49 niletts_llm_qa_train.csv
-rw------- 1 root root 813K May 19 01:49 niletts_llm_qa_val.csv
-rw------- 1 root root 671K May 19 01:49 niletts_test_asr.csv


In [ ]:
# !nvidia-smi

import shutil
import subprocess

if shutil.which("nvidia-smi"):
    subprocess.run(["nvidia-smi"], check=False)
else:
    print("nvidia-smi is not available in this environment.")

Tue May 19 01:50:02 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.95.05              Driver Version: 580.95.05      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A10                     On  |   00000000:17:00.0 Off |                    0 |
|  0%   32C    P8             15W /  150W |       0MiB /  23028MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

# Fine-tune AraELECTRA Extractive QA for NileTTS QA

This notebook is the extractive-QA improvement after the AraT5 baseline.

Why this model:
- Your Qwen-generated QA pairs are extractive: the answer is a span inside `context_ref`.
- AraT5 is generative and produced weak/free-form answers.
- Extractive QA predicts the start and end position of the answer inside the context, which matches your dataset better.

Expected input files:
- `niletts_llm_qa_train.csv`
- `niletts_llm_qa_val.csv`
- `niletts_llm_qa_test_reference.csv`
- `niletts_test_asr.csv`


In [ ]:
# =====
from pathlib import Path

for candidate in (Path.cwd(), *Path.cwd().parents):
    if (candidate / "requirements-qa.txt").is_file():
        requirements_file = candidate / "requirements-qa.txt"
        break
else:
    raise FileNotFoundError("Could not find requirements-qa.txt")

%pip install -r "{requirements_file}"
# =====


In [4]:
import os
import re
import json
import random
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import torch

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForQuestionAnswering,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer
)
from transformers.trainer_utils import get_last_checkpoint

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA capability:", torch.cuda.get_device_capability(0))


Device: cuda
GPU: NVIDIA A10
CUDA capability: (8, 6)


In [ ]:
# =====
from pathlib import Path

for candidate in (Path.cwd(), *Path.cwd().parents):
    if (
        (candidate / "1_ASR").is_dir()
        and (candidate / "2_QA_GENERATION").is_dir()
        and (candidate / "3_FINE_TUNED_QA_MODEL").is_dir()
    ):
        PROJECT_ROOT = candidate
        break
else:
    raise FileNotFoundError(
        "Open the notebook from inside the project folder."
    )

QA_DIR = PROJECT_ROOT / "2_QA_GENERATION"
MODEL_DIR = PROJECT_ROOT / "3_FINE_TUNED_QA_MODEL"

# TRAIN_FILE = "niletts_llm_qa_train.csv"
# VAL_FILE = "niletts_llm_qa_val.csv"
# TEST_FILE = "niletts_llm_qa_test_reference.csv"
# ASR_FILE = "niletts_test_asr.csv"

TRAIN_FILE = (QA_DIR / "1_train_data" / "niletts_llm_qa_train.csv")
VAL_FILE = (QA_DIR / "2_val_data" / "niletts_llm_qa_val.csv")
TEST_FILE = (QA_DIR / "3_test_data"/ "niletts_llm_qa_test_reference.csv")
ASR_FILE = (PROJECT_ROOT / "1_ASR"/ "niletts_asr_eval_finetuning_transcriptions_metrics.csv")

ARTIFACT_DIR = MODEL_DIR / "2_araelectra_niletts_final_outputs"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

MERGED_TEST_FILE = (
    ARTIFACT_DIR
    / "niletts_extract_qa_test_reference_with_asr_context.csv"
)

for path in [TRAIN_FILE, VAL_FILE, TEST_FILE, ASR_FILE]:
    # if not Path(path).exists():
    if not path.is_file():
        # raise FileNotFoundError(f"Missing file: {path}. Upload it or update the path.")
        raise FileNotFoundError(f"Missing file: {path}")

# print("All files found.")
print("All input files found.")
# =====


All files found.


In [7]:
qa_train_df = pd.read_csv(TRAIN_FILE)
qa_val_df = pd.read_csv(VAL_FILE)
qa_test_df = pd.read_csv(TEST_FILE)
asr_df = pd.read_csv(ASR_FILE)

print("Train:", qa_train_df.shape)
print("Val:", qa_val_df.shape)
print("Test QA:", qa_test_df.shape)
print("ASR eval:", asr_df.shape)

display(qa_train_df.head(2))
display(qa_val_df.head(2))
display(qa_test_df.head(2))
display(asr_df.head(2))


Train: (12877, 18)
Val: (1398, 16)
Test QA: (1554, 16)
ASR eval: (950, 6)


,id,split,part_id,metadata_row_index,qa_train_source_index,audio_file,speaker_name,domain,context_ref,question,answer,answer_start_ref,valid_answer_span_ref,question_type,difficulty,needs_manual_review,question_type_raw,difficulty_raw
0,train_6682_0,train,3,101,6682,wavs/sales_11_00000.wav,SPEAKER_02,sales,أهلاً بيكم معانا. النهاردة هنتكلم في موضوع مهم...,الشراكة هي سلاح زو حدين؟,سلاح زو حدين,236,True,domain_specific,easy,True,domain_specific,easy
1,train_6220_0,train,3,6542,6220,wavs/sales_33_00138.wav,SPEAKER_01,sales,المدينة كلها على بعضها كده رواية معمارية بتحكي...,اللغة التي ذكرت في بداية السياق هي:,أحسن نقدر قيمة الأماكن اللي بنعيش فيها أكتر.,131,True,domain_specific,easy,True,domain_specific,easy


,id,split,qa_val_source_index,audio_file,speaker_name,domain,context_ref,question,answer,answer_start_ref,valid_answer_span_ref,question_type,question_type_raw,difficulty,difficulty_raw,needs_manual_review
0,val_1_0,val,1,wavs/sales_22_00010.wav,SPEAKER_02,sales,للشركة بتصرفاته هو شخصياً. لو هو هادي و بيحترم...,الشركة تتعامل مع العملاء بشكل شخصي؟,لو هو هادي و بيحترم الناس و بيسمع الفريق بيقلده.,27,True,domain_specific,domain_specific,easy,easy,True
1,val_1_1,val,1,wavs/sales_22_00010.wav,SPEAKER_02,sales,للشركة بتصرفاته هو شخصياً. لو هو هادي و بيحترم...,كيف يمكن أن يتعلم الفريق من الشخص العصبي؟,لو هو عصبي و بيلوم على طول الفريق بيتعلم يخاف ...,76,True,detail_extraction,detail_extraction,medium,medium,True


,id,split,qa_test_source_index,audio_file,speaker_name,domain,context_ref,question,answer,answer_start_ref,valid_answer_span_ref,question_type,question_type_raw,difficulty,difficulty_raw,needs_manual_review
0,test_0_0,test,0,wavs/sales_34_00019.wav,SPEAKER_02,sales,فالشخص اللي هو يعني حافز مش فاهم ده. اللي قافل...,الشخص الذي يُعتبر حافزًا في هذا السياق هو من ي...,الشخص اللي هو يعني حافز مش فاهم ده.,1,True,domain_specific,domain_specific,easy,easy,True
1,test_0_1,test,0,wavs/sales_34_00019.wav,SPEAKER_02,sales,فالشخص اللي هو يعني حافز مش فاهم ده. اللي قافل...,ما هي المشكلة التي قد تنشأ بسبب الفضول؟,الأهم دلوقتي الشخص اللي عنده فضول.,105,True,intent,intent/main_idea,medium,medium,True


,audio_file,speaker_name,reference_text,asr_text,wer,cer
0,wavs/sales_34_00019.wav,SPEAKER_02,فالشخص اللي هو يعني حافز مش فاهم ده. اللي قافل...,فالشخص اللي هو يعني حافظ مش فاهم ده. اللي قافل...,0.035714,0.007407
1,wavs/sales_26_00082.wav,SPEAKER_01,نكون خدنا جولة سريعة و مكثفة يعني في أهم الأفك...,نكون خدنا جولة سريعة و مكثفة يعني في أهم الأفك...,0.000000,0.000000


In [8]:
def normalize_spaces(text):
    return re.sub(r"\s+", " ", str(text)).strip()


def find_answer_start_soft(context, answer):
    context = str(context)
    answer = str(answer)

    exact = context.find(answer)
    if exact != -1:
        return exact

    context_norm = normalize_spaces(context)
    answer_norm = normalize_spaces(answer)
    norm_pos = context_norm.find(answer_norm)

    return norm_pos if norm_pos != -1 else -1


def validate_qa_df(df, name):
    required = [
        "audio_file", "speaker_name", "domain", "context_ref",
        "question", "answer"
    ]

    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"{name} missing columns: {missing}")

    out = df.copy()

    for col in ["audio_file", "speaker_name", "domain", "context_ref", "question", "answer"]:
        out[col] = out[col].astype(str).map(normalize_spaces)

    out["answer_start_ref"] = out.apply(
        lambda r: find_answer_start_soft(r["context_ref"], r["answer"]),
        axis=1
    )

    out["valid_answer_span_ref"] = out["answer_start_ref"] != -1

    before = len(out)

    out = out[out["question"].str.len() > 5]
    out = out[out["answer"].str.len() > 0]
    out = out[out["valid_answer_span_ref"] == True]
    out = out.drop_duplicates(subset=["audio_file", "question", "answer"]).reset_index(drop=True)

    print(f"{name}: kept {len(out)} / {before}")
    print(f"{name}: unique audio files = {out['audio_file'].nunique()}")

    return out


qa_train_df = validate_qa_df(qa_train_df, "train")
qa_val_df = validate_qa_df(qa_val_df, "val")
qa_test_df = validate_qa_df(qa_test_df, "test")

required_asr = ["audio_file", "speaker_name", "reference_text", "asr_text", "wer", "cer"]
missing_asr = [c for c in required_asr if c not in asr_df.columns]

if missing_asr:
    raise ValueError(f"ASR file missing columns: {missing_asr}")

asr_df = asr_df.copy()

for col in ["audio_file", "speaker_name", "reference_text", "asr_text"]:
    asr_df[col] = asr_df[col].astype(str).map(normalize_spaces)

asr_df["wer"] = pd.to_numeric(asr_df["wer"], errors="coerce")
asr_df["cer"] = pd.to_numeric(asr_df["cer"], errors="coerce")

print("ASR WER:", asr_df["wer"].mean())
print("ASR CER:", asr_df["cer"].mean())


train: kept 12877 / 12877
train: unique audio files = 5962
val: kept 1398 / 1398
val: unique audio files = 665
test: kept 1554 / 1554
test: unique audio files = 738
ASR WER: 0.11683900680229063
ASR CER: 0.038978167019728595


In [9]:
def show_split_stats(name, df, source_col=None):
    print("\n" + "="*40)
    print(name)
    print("Rows:", len(df))
    print("Unique audio files:", df["audio_file"].nunique())

    if source_col and source_col in df.columns:
        print(f"Unique {source_col}:", df[source_col].nunique())

    print("Domain distribution:")
    display(df["domain"].value_counts())

    if "question_type" in df.columns:
        print("Question type distribution:")
        display(df["question_type"].value_counts(dropna=False))


show_split_stats("Train", qa_train_df, "qa_train_source_index")
show_split_stats("Val", qa_val_df, "qa_val_source_index")
show_split_stats("Test", qa_test_df, "qa_test_source_index")

train_audio = set(qa_train_df["audio_file"])
val_audio = set(qa_val_df["audio_file"])
test_audio = set(qa_test_df["audio_file"])

print("\nSplit overlap checks:")
print("Train ∩ Val:", len(train_audio & val_audio))
print("Train ∩ Test:", len(train_audio & test_audio))
print("Val ∩ Test:", len(val_audio & test_audio))

asr_audio = set(asr_df["audio_file"])

print("\nASR/test matching:")
print("Test QA audio missing in ASR:", len(test_audio - asr_audio))
print("ASR audio not used by filtered QA test:", len(asr_audio - test_audio))



Train
Rows: 12877
Unique audio files: 5962
Unique qa_train_source_index: 5962
Domain distribution:


domain
sales      6984
general    4009
medical    1884
Name: count, dtype: int64

Question type distribution:


question_type
detail_extraction    4621
domain_specific      4503
intent               2322
main_idea            1168
number_extraction     263
Name: count, dtype: int64


Val
Rows: 1398
Unique audio files: 665
Unique qa_val_source_index: 665
Domain distribution:


...


Test
Rows: 1554
Unique audio files: 738
Unique qa_test_source_index: 738
Domain distribution:


domain
sales      847
general    495
medical    212
Name: count, dtype: int64

Question type distribution:


question_type
detail_extraction    551
domain_specific      492
intent               322
main_idea            162
number_extraction     27
Name: count, dtype: int64


Split overlap checks:
Train ∩ Val: 0
Train ∩ Test: 0
Val ∩ Test: 0

ASR/test matching:
Test QA audio missing in ASR: 0
ASR audio not used by filtered QA test: 212


In [ ]:
qa_test_with_asr_df = qa_test_df.merge(
    asr_df[["audio_file", "asr_text", "wer", "cer", "reference_text"]],
    on="audio_file",
    how="left"
).rename(columns={
    "asr_text": "context_asr",
    "wer": "asr_wer",
    "cer": "asr_cer"
})

missing_asr_context = qa_test_with_asr_df["context_asr"].isna().sum()

print("Missing ASR contexts after merge:", missing_asr_context)

if missing_asr_context > 0:
    raise ValueError("Some test QA rows do not have ASR transcripts. Check audio_file matching.")

qa_test_with_asr_df["answer_in_asr_exact"] = qa_test_with_asr_df.apply(
    lambda r: str(r["answer"]) in str(r["context_asr"]),
    axis=1
)

print("Test QA rows:", len(qa_test_with_asr_df))
print("Unique test audios:", qa_test_with_asr_df["audio_file"].nunique())
print("Answer appears exactly inside ASR context:", qa_test_with_asr_df["answer_in_asr_exact"].mean())
print("Mean ASR WER over QA rows:", qa_test_with_asr_df["asr_wer"].mean())
print("Mean ASR CER over QA rows:", qa_test_with_asr_df["asr_cer"].mean())

# =====
# qa_test_with_asr_df.to_csv(
#     "niletts_extract_qa_test_reference_with_asr_context.csv",
#     index=False,
#     encoding="utf-8-sig"
# )

qa_test_with_asr_df.to_csv(
    MERGED_TEST_FILE,
    index=False,
    encoding="utf-8-sig"
)
# =====

display(qa_test_with_asr_df.head(2))


Missing ASR contexts after merge: 0
Test QA rows: 1554
Unique test audios: 738
Answer appears exactly inside ASR context: 0.5096525096525096
Mean ASR WER over QA rows: 0.11756393013612884
Mean ASR CER over QA rows: 0.04121027254249641


,id,split,qa_test_source_index,audio_file,speaker_name,domain,context_ref,question,answer,answer_start_ref,...,question_type,question_type_raw,difficulty,difficulty_raw,needs_manual_review,context_asr,asr_wer,asr_cer,reference_text,answer_in_asr_exact
0,test_0_0,test,0,wavs/sales_34_00019.wav,SPEAKER_02,sales,فالشخص اللي هو يعني حافز مش فاهم ده. اللي قافل...,الشخص الذي يُعتبر حافزًا في هذا السياق هو من ي...,الشخص اللي هو يعني حافز مش فاهم ده.,1,...,domain_specific,domain_specific,easy,easy,True,فالشخص اللي هو يعني حافظ مش فاهم ده. اللي قافل...,0.035714,0.007407,فالشخص اللي هو يعني حافز مش فاهم ده. اللي قافل...,False
1,test_0_1,test,0,wavs/sales_34_00019.wav,SPEAKER_02,sales,فالشخص اللي هو يعني حافز مش فاهم ده. اللي قافل...,ما هي المشكلة التي قد تنشأ بسبب الفضول؟,الأهم دلوقتي الشخص اللي عنده فضول.,105,...,intent,intent/main_idea,medium,medium,True,فالشخص اللي هو يعني حافظ مش فاهم ده. اللي قافل...,0.035714,0.007407,فالشخص اللي هو يعني حافز مش فاهم ده. اللي قافل...,True


In [11]:
MODEL_NAME = "ZeyadAhmed/AraElectra-Arabic-SQuADv2-QA"

MAX_LENGTH = 384
DOC_STRIDE = 128
MAX_ANSWER_LENGTH = 50

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
model = AutoModelForQuestionAnswering.from_pretrained(MODEL_NAME)
model.to(device)

print("Loaded:", MODEL_NAME)
print("Tokenizer fast:", tokenizer.is_fast)


tokenizer_config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

/usr/local/lib/python3.12/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


config.json:   0%|          | 0.00/851 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/538M [00:00<?, ?B/s]

Loaded: ZeyadAhmed/AraElectra-Arabic-SQuADv2-QA
Tokenizer fast: True


In [12]:
def prepare_train_features(df, context_col="context_ref"):
    questions = df["question"].astype(str).tolist()
    contexts = df[context_col].astype(str).tolist()

    tokenized = tokenizer(
        questions,
        contexts,
        truncation="only_second",
        max_length=MAX_LENGTH,
        stride=DOC_STRIDE,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length"
    )

    sample_mapping = tokenized.pop("overflow_to_sample_mapping")
    offset_mapping = tokenized.pop("offset_mapping")

    start_positions = []
    end_positions = []
    valid_answer_feature_count = 0

    for i, offsets in enumerate(offset_mapping):
        sample_idx = sample_mapping[i]
        row = df.iloc[sample_idx]

        answer = str(row["answer"])
        answer_start = int(row["answer_start_ref"])
        answer_end = answer_start + len(answer)

        input_ids = tokenized["input_ids"][i]
        cls_index = input_ids.index(tokenizer.cls_token_id)

        sequence_ids = tokenized.sequence_ids(i)

        token_start_index = 0
        while token_start_index < len(sequence_ids) and sequence_ids[token_start_index] != 1:
            token_start_index += 1

        token_end_index = len(sequence_ids) - 1
        while token_end_index >= 0 and sequence_ids[token_end_index] != 1:
            token_end_index -= 1

        if (
            token_start_index >= len(offsets)
            or token_end_index < 0
            or offsets[token_start_index][0] > answer_start
            or offsets[token_end_index][1] < answer_end
        ):
            start_positions.append(cls_index)
            end_positions.append(cls_index)
        else:
            while (
                token_start_index < len(offsets)
                and offsets[token_start_index][0] <= answer_start
            ):
                token_start_index += 1

            start_pos = token_start_index - 1

            while offsets[token_end_index][1] >= answer_end:
                token_end_index -= 1

            end_pos = token_end_index + 1

            start_positions.append(start_pos)
            end_positions.append(end_pos)
            valid_answer_feature_count += 1

    tokenized["start_positions"] = start_positions
    tokenized["end_positions"] = end_positions

    print("Features:", len(start_positions))
    print("Features containing the gold answer:", valid_answer_feature_count)

    return Dataset.from_dict(tokenized)


tokenized_train = prepare_train_features(qa_train_df, context_col="context_ref")
tokenized_val = prepare_train_features(qa_val_df, context_col="context_ref")

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

print(tokenized_train)
print(tokenized_val)


Features: 12877
Features containing the gold answer: 12877
Features: 1398
Features containing the gold answer: 1398
Dataset({
    features: ['input_ids', 'token_type_ids', 'attention_mask', 'start_positions', 'end_positions'],
    num_rows: 12877
})
Dataset({
    features: ['input_ids', 'token_type_ids', 'attention_mask', 'start_positions', 'end_positions'],
    num_rows: 1398
})


In [13]:
def prepare_eval_features(df, context_col):
    questions = df["question"].astype(str).tolist()
    contexts = df[context_col].astype(str).tolist()

    tokenized = tokenizer(
        questions,
        contexts,
        truncation="only_second",
        max_length=MAX_LENGTH,
        stride=DOC_STRIDE,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length"
    )

    sample_mapping = tokenized.pop("overflow_to_sample_mapping")
    offset_mapping = tokenized.pop("offset_mapping")

    model_features = []
    metadata_features = []

    for i, offsets in enumerate(offset_mapping):
        sample_idx = int(sample_mapping[i])
        sequence_ids = tokenized.sequence_ids(i)

        context_offsets = []
        for k, offset in enumerate(offsets):
            if sequence_ids[k] == 1:
                context_offsets.append(offset)
            else:
                context_offsets.append(None)

        model_feature = {}
        for key in tokenized.keys():
            model_feature[key] = tokenized[key][i]

        metadata_features.append({
            "example_id": sample_idx,
            "offset_mapping": context_offsets
        })

        model_features.append(model_feature)

    model_dataset = Dataset.from_list(model_features)

    print(f"{context_col} examples:", len(df))
    print(f"{context_col} features:", len(model_dataset))

    return model_dataset, metadata_features


tokenized_test_ref, test_ref_metadata = prepare_eval_features(
    qa_test_with_asr_df,
    context_col="context_ref"
)

tokenized_test_asr, test_asr_metadata = prepare_eval_features(
    qa_test_with_asr_df,
    context_col="context_asr"
)


context_ref examples: 1554
context_ref features: 1554
context_asr examples: 1554
context_asr features: 1554


In [14]:
AR_DIACRITICS = re.compile(r"[\u0617-\u061A\u064B-\u0652\u0670]")
PUNCT = re.compile(r"[^\w\s\u0600-\u06FF]")


def normalize_arabic(text):
    text = str(text)
    text = AR_DIACRITICS.sub("", text)
    text = text.replace("ـ", "")
    text = re.sub(r"[إأآا]", "ا", text)
    text = text.replace("ى", "ي")
    text = text.replace("ة", "ه")
    text = PUNCT.sub(" ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def exact_match_score(prediction, ground_truth):
    return int(normalize_arabic(prediction) == normalize_arabic(ground_truth))


def token_f1_score(prediction, ground_truth):
    pred_tokens = normalize_arabic(prediction).split()
    gold_tokens = normalize_arabic(ground_truth).split()

    if len(pred_tokens) == 0 and len(gold_tokens) == 0:
        return 1.0

    if len(pred_tokens) == 0 or len(gold_tokens) == 0:
        return 0.0

    common = {}
    for tok in set(pred_tokens):
        common[tok] = min(pred_tokens.count(tok), gold_tokens.count(tok))

    num_same = sum(common.values())

    if num_same == 0:
        return 0.0

    precision = num_same / len(pred_tokens)
    recall = num_same / len(gold_tokens)

    return 2 * precision * recall / (precision + recall)


def postprocess_qa_predictions(examples_df, features_metadata, predictions, context_col,
                               n_best_size=20, max_answer_length=MAX_ANSWER_LENGTH):
    start_logits, end_logits = predictions

    example_to_features = defaultdict(list)

    for feature_idx, meta in enumerate(features_metadata):
        example_to_features[meta["example_id"]].append(feature_idx)

    predicted_answers = []

    for example_id, row in examples_df.reset_index(drop=True).iterrows():
        context = str(row[context_col])
        valid_answers = []

        for feature_idx in example_to_features[example_id]:
            start_logit = start_logits[feature_idx]
            end_logit = end_logits[feature_idx]
            offsets = features_metadata[feature_idx]["offset_mapping"]

            start_indexes = np.argsort(start_logit)[-n_best_size:][::-1]
            end_indexes = np.argsort(end_logit)[-n_best_size:][::-1]

            for start_index in start_indexes:
                for end_index in end_indexes:
                    if start_index >= len(offsets) or end_index >= len(offsets):
                        continue
                    if offsets[start_index] is None or offsets[end_index] is None:
                        continue
                    if end_index < start_index:
                        continue

                    length = end_index - start_index + 1

                    if length > max_answer_length:
                        continue

                    start_char = offsets[start_index][0]
                    end_char = offsets[end_index][1]

                    if start_char is None or end_char is None:
                        continue

                    text = context[start_char:end_char]

                    valid_answers.append({
                        "score": float(start_logit[start_index] + end_logit[end_index]),
                        "text": text
                    })

        if valid_answers:
            best_answer = max(valid_answers, key=lambda x: x["score"])["text"]
        else:
            best_answer = ""

        predicted_answers.append(best_answer)

    return predicted_answers


def compute_qa_metrics(predictions, references):
    em_scores = [exact_match_score(p, g) for p, g in zip(predictions, references)]
    f1_scores = [token_f1_score(p, g) for p, g in zip(predictions, references)]

    return {
        "exact_match": float(np.mean(em_scores)),
        "f1": float(np.mean(f1_scores))
    }


In [ ]:
# =====
# OUTPUT_DIR = "araelectra_niletts_extractive_qa"
# BEST_MODEL_DIR = "araelectra_niletts_extractive_qa_best"

OUTPUT_DIR = str(
    MODEL_DIR / "araelectra_niletts_extractive_qa"
)

BEST_MODEL_DIR = str(
    ARTIFACT_DIR / "araelectra_niletts_extractive_qa_best"
)
# =====

PER_DEVICE_TRAIN_BATCH_SIZE = 8
PER_DEVICE_EVAL_BATCH_SIZE = 16
GRADIENT_ACCUMULATION_STEPS = 1

NUM_EPOCHS = 3
LEARNING_RATE = 2e-5
SAVE_EVERY_STEPS = 500
EVAL_EVERY_STEPS = 500
LOGGING_STEPS = 50

if torch.cuda.is_available():
    major, minor = torch.cuda.get_device_capability(0)
    USE_BF16 = major >= 8
    USE_FP16 = not USE_BF16
else:
    USE_BF16 = False
    USE_FP16 = False

print("USE_BF16:", USE_BF16)
print("USE_FP16:", USE_FP16)
print("Effective train batch size:", PER_DEVICE_TRAIN_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS)


USE_BF16: True
USE_FP16: False
Effective train batch size: 8


In [16]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    overwrite_output_dir=False,

    learning_rate=LEARNING_RATE,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=PER_DEVICE_EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    weight_decay=0.01,

    eval_strategy="steps",
    eval_steps=EVAL_EVERY_STEPS,
    save_strategy="steps",
    save_steps=SAVE_EVERY_STEPS,
    logging_strategy="steps",
    logging_steps=LOGGING_STEPS,

    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    save_total_limit=3,

    fp16=USE_FP16,
    bf16=USE_BF16,

    save_safetensors=False,

    report_to="none",
    seed=SEED,
    dataloader_num_workers=0,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    tokenizer=tokenizer,
    data_collator=data_collator,
)


Detected kernel version 4.4.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.


In [17]:
last_checkpoint = None

if Path(OUTPUT_DIR).exists():
    last_checkpoint = get_last_checkpoint(OUTPUT_DIR)

print("Last checkpoint:", last_checkpoint)

train_result = trainer.train(resume_from_checkpoint=last_checkpoint)

trainer.save_model(BEST_MODEL_DIR)
tokenizer.save_pretrained(BEST_MODEL_DIR)

metrics = train_result.metrics
trainer.log_metrics("train", metrics)
trainer.save_metrics("train", metrics)
trainer.save_state()

print("Saved best model to:", BEST_MODEL_DIR)


Last checkpoint: None


Step,Training Loss,Validation Loss
500,1.821000,1.800833
1000,1.802800,1.744684
1500,1.731100,1.744029
2000,1.411500,1.752356
2500,1.402500,1.790144
3000,1.299800,1.791963
3500,1.013600,1.971528
4000,1.050300,1.931278
4500,0.985400,1.963622


***** train metrics *****
  epoch                    =        3.0
  total_flos               =  7050685GF
  train_loss               =     1.3985
  train_runtime            = 0:10:37.00
  train_samples_per_second =     60.645
  train_steps_per_second   =      7.582
Saved best model to: araelectra_niletts_extractive_qa_best


In [ ]:
ref_pred_output = trainer.predict(tokenized_test_ref)
ref_predictions = ref_pred_output.predictions

ref_pred_texts = postprocess_qa_predictions(
    examples_df=qa_test_with_asr_df,
    features_metadata=test_ref_metadata,
    predictions=ref_predictions,
    context_col="context_ref"
)

gold_texts = qa_test_with_asr_df["answer"].astype(str).tolist()

ref_metrics = compute_qa_metrics(ref_pred_texts, gold_texts)

eval_ref_df = qa_test_with_asr_df.copy()
eval_ref_df["gold_answer"] = gold_texts
eval_ref_df["pred_answer_ref_context"] = ref_pred_texts

eval_ref_df["em_ref"] = [exact_match_score(p, g) for p, g in zip(ref_pred_texts, gold_texts)]
eval_ref_df["f1_ref"] = [token_f1_score(p, g) for p, g in zip(ref_pred_texts, gold_texts)]

print("Reference Transcript QA Exact Match:", ref_metrics["exact_match"])
print("Reference Transcript QA F1:", ref_metrics["f1"])

# =====
eval_ref_df.to_csv(
    "niletts_araelectra_reference_eval_predictions.csv",
    index=False,
    encoding="utf-8-sig"
)

eval_ref_df.to_csv(
    ARTIFACT_DIR / "niletts_araelectra_reference_eval_predictions.csv",
    index=False,
    encoding="utf-8-sig"
)
# =====

display(eval_ref_df[[
    "audio_file", "domain", "question", "context_ref",
    "gold_answer", "pred_answer_ref_context", "em_ref", "f1_ref"
]].head())


Reference Transcript QA Exact Match: 0.2876447876447876
Reference Transcript QA F1: 0.5557779358974736


,audio_file,domain,question,context_ref,gold_answer,pred_answer_ref_context,em_ref,f1_ref
0,wavs/sales_34_00019.wav,sales,الشخص الذي يُعتبر حافزًا في هذا السياق هو من ي...,فالشخص اللي هو يعني حافز مش فاهم ده. اللي قافل...,الشخص اللي هو يعني حافز مش فاهم ده.,حافز مش فاهم ده.,0,0.666667
1,wavs/sales_34_00019.wav,sales,ما هي المشكلة التي قد تنشأ بسبب الفضول؟,فالشخص اللي هو يعني حافز مش فاهم ده. اللي قافل...,الأهم دلوقتي الشخص اللي عنده فضول.,ده ممكن يبقى مشكلة بعدين.,0,0.000000
2,wavs/sales_34_00019.wav,sales,هل هناك أي مشكلة معقدة يمكن أن تنشأ بسبب الفضول؟,فالشخص اللي هو يعني حافز مش فاهم ده. اللي قافل...,ده ممكن يبقى مشكلة بعدين.,ده ممكن يبقى مشكلة بعدين.,1,1.000000
3,wavs/sales_26_00082.wav,sales,ما هو الموضوع الأهم الذي سيتم التطرق إليه خلال...,نكون خدنا جولة سريعة و مكثفة يعني في أهم الأفك...,بناء فريق عمل ناجح.,بناء فريق عمل ناجح.,1,1.000000
4,wavs/sales_26_00082.wav,sales,هل هناك أي موضوع آخر سيُستكشف بعد ذلك؟,نكون خدنا جولة سريعة و مكثفة يعني في أهم الأفك...,موضوع تاني لا يقل أهمية أبداً.,المرة الجاية إن شاء الله هنستكشف مع بعض موضوع ...,0,0.600000


In [ ]:
asr_pred_output = trainer.predict(tokenized_test_asr)
asr_predictions = asr_pred_output.predictions

asr_pred_texts = postprocess_qa_predictions(
    examples_df=qa_test_with_asr_df,
    features_metadata=test_asr_metadata,
    predictions=asr_predictions,
    context_col="context_asr"
)

gold_texts = qa_test_with_asr_df["answer"].astype(str).tolist()

asr_metrics = compute_qa_metrics(asr_pred_texts, gold_texts)

eval_asr_df = qa_test_with_asr_df.copy()
eval_asr_df["gold_answer"] = gold_texts
eval_asr_df["pred_answer_asr_context"] = asr_pred_texts

eval_asr_df["em_asr"] = [exact_match_score(p, g) for p, g in zip(asr_pred_texts, gold_texts)]
eval_asr_df["f1_asr"] = [token_f1_score(p, g) for p, g in zip(asr_pred_texts, gold_texts)]

print("ASR Transcript QA Exact Match:", asr_metrics["exact_match"])
print("ASR Transcript QA F1:", asr_metrics["f1"])

# =====
# eval_asr_df.to_csv(
#     "niletts_araelectra_asr_eval_predictions.csv",
#     index=False,
#     encoding="utf-8-sig"
# )

eval_asr_df.to_csv(
    ARTIFACT_DIR / "niletts_araelectra_asr_eval_predictions.csv",
    index=False,
    encoding="utf-8-sig"
)
# =====

display(eval_asr_df[[
    "audio_file", "domain", "asr_wer", "asr_cer", "question", "context_asr",
    "gold_answer", "pred_answer_asr_context", "em_asr", "f1_asr"
]].head())


ASR Transcript QA Exact Match: 0.16216216216216217
ASR Transcript QA F1: 0.4857671819180413


,audio_file,domain,asr_wer,asr_cer,question,context_asr,gold_answer,pred_answer_asr_context,em_asr,f1_asr
0,wavs/sales_34_00019.wav,sales,0.035714,0.007407,الشخص الذي يُعتبر حافزًا في هذا السياق هو من ي...,فالشخص اللي هو يعني حافظ مش فاهم ده. اللي قافل...,الشخص اللي هو يعني حافز مش فاهم ده.,حافظ مش فاهم ده.,0,0.5
1,wavs/sales_34_00019.wav,sales,0.035714,0.007407,ما هي المشكلة التي قد تنشأ بسبب الفضول؟,فالشخص اللي هو يعني حافظ مش فاهم ده. اللي قافل...,الأهم دلوقتي الشخص اللي عنده فضول.,ده ممكن يبقى مشكلة بعدين.,0,0.0
2,wavs/sales_34_00019.wav,sales,0.035714,0.007407,هل هناك أي مشكلة معقدة يمكن أن تنشأ بسبب الفضول؟,فالشخص اللي هو يعني حافظ مش فاهم ده. اللي قافل...,ده ممكن يبقى مشكلة بعدين.,ده ممكن يبقى مشكلة بعدين.,1,1.0
3,wavs/sales_26_00082.wav,sales,0.000000,0.000000,ما هو الموضوع الأهم الذي سيتم التطرق إليه خلال...,نكون خدنا جولة سريعة و مكثفة يعني في أهم الأفك...,بناء فريق عمل ناجح.,بناء فريق عمل ناجح.,1,1.0
4,wavs/sales_26_00082.wav,sales,0.000000,0.000000,هل هناك أي موضوع آخر سيُستكشف بعد ذلك؟,نكون خدنا جولة سريعة و مكثفة يعني في أهم الأفك...,موضوع تاني لا يقل أهمية أبداً.,المرة الجاية إن شاء الله هنستكشف مع بعض موضوع ...,0,0.6


In [ ]:
comparison_df = eval_ref_df[[
    "audio_file",
    "domain",
    "question",
    "answer",
    "context_ref",
    "context_asr",
    "asr_wer",
    "asr_cer",
    "answer_in_asr_exact",
    "pred_answer_ref_context",
    "em_ref",
    "f1_ref",
]].copy()

comparison_df["pred_answer_asr_context"] = eval_asr_df["pred_answer_asr_context"].values
comparison_df["em_asr"] = eval_asr_df["em_asr"].values
comparison_df["f1_asr"] = eval_asr_df["f1_asr"].values

comparison_df["em_drop"] = comparison_df["em_ref"] - comparison_df["em_asr"]
comparison_df["f1_drop"] = comparison_df["f1_ref"] - comparison_df["f1_asr"]

summary_metrics = {
    "model": MODEL_NAME,
    "model_type": "extractive_qa",
    "reference_em": float(comparison_df["em_ref"].mean()),
    "reference_f1": float(comparison_df["f1_ref"].mean()),
    "asr_em": float(comparison_df["em_asr"].mean()),
    "asr_f1": float(comparison_df["f1_asr"].mean()),
    "em_drop": float(comparison_df["em_drop"].mean()),
    "f1_drop": float(comparison_df["f1_drop"].mean()),
    "asr_wer_mean": float(comparison_df["asr_wer"].mean()),
    "asr_cer_mean": float(comparison_df["asr_cer"].mean()),
    "answer_in_asr_exact_rate": float(comparison_df["answer_in_asr_exact"].mean()),
    "num_test_qa_pairs": int(len(comparison_df)),
    "num_test_audio_files": int(comparison_df["audio_file"].nunique()),
}

print(json.dumps(summary_metrics, ensure_ascii=False, indent=2))

# =====
# with open("niletts_araelectra_reference_vs_asr_summary_metrics.json", "w", encoding="utf-8") as f:
with open(ARTIFACT_DIR / "niletts_araelectra_reference_vs_asr_summary_metrics.json", "w", encoding="utf-8") as f:
    json.dump(summary_metrics, f, ensure_ascii=False, indent=2)
# =====

# =====
# comparison_df.to_csv(
#     "niletts_araelectra_reference_vs_asr_comparison.csv",
#     index=False,
#     encoding="utf-8-sig"
# )

comparison_df.to_csv(
    ARTIFACT_DIR / "niletts_araelectra_reference_vs_asr_comparison.csv",
    index=False,
    encoding="utf-8-sig"
)
# =====

display(comparison_df.head())


{
  "model": "ZeyadAhmed/AraElectra-Arabic-SQuADv2-QA",
  "model_type": "extractive_qa",
  "reference_em": 0.2876447876447876,
  "reference_f1": 0.5557779358974736,
  "asr_em": 0.16216216216216217,
  "asr_f1": 0.4857671819180413,
  "em_drop": 0.12548262548262548,
  "f1_drop": 0.07001075397943218,
  "asr_wer_mean": 0.11756393013612884,
  "asr_cer_mean": 0.04121027254249641,
  "answer_in_asr_exact_rate": 0.5096525096525096,
  "num_test_qa_pairs": 1554,
  "num_test_audio_files": 738
}


,audio_file,domain,question,answer,context_ref,context_asr,asr_wer,asr_cer,answer_in_asr_exact,pred_answer_ref_context,em_ref,f1_ref,pred_answer_asr_context,em_asr,f1_asr,em_drop,f1_drop
0,wavs/sales_34_00019.wav,sales,الشخص الذي يُعتبر حافزًا في هذا السياق هو من ي...,الشخص اللي هو يعني حافز مش فاهم ده.,فالشخص اللي هو يعني حافز مش فاهم ده. اللي قافل...,فالشخص اللي هو يعني حافظ مش فاهم ده. اللي قافل...,0.035714,0.007407,False,حافز مش فاهم ده.,0,0.666667,حافظ مش فاهم ده.,0,0.5,0,0.166667
1,wavs/sales_34_00019.wav,sales,ما هي المشكلة التي قد تنشأ بسبب الفضول؟,الأهم دلوقتي الشخص اللي عنده فضول.,فالشخص اللي هو يعني حافز مش فاهم ده. اللي قافل...,فالشخص اللي هو يعني حافظ مش فاهم ده. اللي قافل...,0.035714,0.007407,True,ده ممكن يبقى مشكلة بعدين.,0,0.000000,ده ممكن يبقى مشكلة بعدين.,0,0.0,0,0.000000
2,wavs/sales_34_00019.wav,sales,هل هناك أي مشكلة معقدة يمكن أن تنشأ بسبب الفضول؟,ده ممكن يبقى مشكلة بعدين.,فالشخص اللي هو يعني حافز مش فاهم ده. اللي قافل...,فالشخص اللي هو يعني حافظ مش فاهم ده. اللي قافل...,0.035714,0.007407,True,ده ممكن يبقى مشكلة بعدين.,1,1.000000,ده ممكن يبقى مشكلة بعدين.,1,1.0,0,0.000000
3,wavs/sales_26_00082.wav,sales,ما هو الموضوع الأهم الذي سيتم التطرق إليه خلال...,بناء فريق عمل ناجح.,نكون خدنا جولة سريعة و مكثفة يعني في أهم الأفك...,نكون خدنا جولة سريعة و مكثفة يعني في أهم الأفك...,0.000000,0.000000,True,بناء فريق عمل ناجح.,1,1.000000,بناء فريق عمل ناجح.,1,1.0,0,0.000000
4,wavs/sales_26_00082.wav,sales,هل هناك أي موضوع آخر سيُستكشف بعد ذلك؟,موضوع تاني لا يقل أهمية أبداً.,نكون خدنا جولة سريعة و مكثفة يعني في أهم الأفك...,نكون خدنا جولة سريعة و مكثفة يعني في أهم الأفك...,0.000000,0.000000,True,المرة الجاية إن شاء الله هنستكشف مع بعض موضوع ...,0,0.600000,المرة الجاية إن شاء الله هنستكشف مع بعض موضوع ...,0,0.6,0,0.000000


In [ ]:
domain_summary = comparison_df.groupby("domain").agg(
    qa_pairs=("audio_file", "count"),
    unique_audio=("audio_file", "nunique"),
    mean_asr_wer=("asr_wer", "mean"),
    mean_asr_cer=("asr_cer", "mean"),
    reference_em=("em_ref", "mean"),
    reference_f1=("f1_ref", "mean"),
    asr_em=("em_asr", "mean"),
    asr_f1=("f1_asr", "mean"),
    mean_f1_drop=("f1_drop", "mean"),
    answer_in_asr_exact_rate=("answer_in_asr_exact", "mean"),
).reset_index()

# =====
# domain_summary.to_csv(
#     "niletts_araelectra_domain_level_results.csv",
#     index=False,
#     encoding="utf-8-sig"
# )

domain_summary.to_csv(
    ARTIFACT_DIR / "niletts_araelectra_domain_level_results.csv",
    index=False,
    encoding="utf-8-sig"
)
# =====

display(domain_summary)


,domain,qa_pairs,unique_audio,mean_asr_wer,mean_asr_cer,reference_em,reference_f1,asr_em,asr_f1,mean_f1_drop,answer_in_asr_exact_rate
0,general,495,233,0.097940,0.033252,0.234343,0.540250,0.149495,0.486610,0.053639,0.541414
1,medical,212,101,0.140479,0.075467,0.221698,0.550258,0.150943,0.502117,0.048142,0.584906
2,sales,847,404,0.123297,0.037287,0.335301,0.566234,0.172373,0.481182,0.085052,0.472255


In [ ]:
worst_examples = comparison_df.sort_values(
    by=["f1_drop", "asr_wer"],
    ascending=[False, False]
).head(50)

# =====
# worst_examples.to_csv(
#     "niletts_araelectra_worst_asr_error_examples.csv",
#     index=False,
#     encoding="utf-8-sig"
# )

worst_examples.to_csv(
    ARTIFACT_DIR / "niletts_araelectra_worst_asr_error_examples.csv",
    index=False,
    encoding="utf-8-sig"
)
# =====

display(worst_examples[[
    "audio_file", "domain", "asr_wer", "question", "answer",
    "pred_answer_ref_context", "pred_answer_asr_context",
    "f1_ref", "f1_asr", "f1_drop",
    "context_ref", "context_asr"
]].head(10))


,audio_file,domain,asr_wer,question,answer,pred_answer_ref_context,pred_answer_asr_context,f1_ref,f1_asr,f1_drop,context_ref,context_asr
326,wavs/general_4_00056.wav,general,1.000000,من الذي قام بترجمة هذا الفيديو؟,نانسي قنقر,نانسي قنقر,,1.0,0.0,1.0,ترجمة نانسي قنقر,وكسر
1149,wavs/sales_15_00041.wav,sales,0.483871,هل هذه هي واحدة من أفضل الأمثلة التي تم تقديمه...,هي دي أشهر مثال لها.,هي دي أشهر مثال لها.,"99, 199, 495.",1.0,0.0,1.0,آه بتاع التسعة و تسعين جنيه بدل مية. هي دي أشه...,آه بتاع 99 جنيه بدل 100. هي دي أشهر مثال لها 9...
1455,wavs/sales_22_00065.wav,sales,0.228571,ما هو الموضوع الرئيسي للنقاش الأول؟,التطور الكبير في طريقة البحث عن عقار.,التطور الكبير في طريقة البحث عن عقار.,بأول نقطة اللي هي ممكن أكتر حاجة لمسنها كلنا.,1.0,0.0,1.0,محاور كلها شيقة و مهمة جداً. خلينا نبدأ بأول ن...,محاور كلها الشيقة و مهمة جداً. خلينا نبدأ بأول...
721,wavs/sales_6_00103.wav,sales,0.219512,من هو المصدر الذي يوفر الرشوة الممتازة والمفيدة؟,المصدر مديلنا رشته صريغة ومفيدة.,المصدر مديلنا رشته صريغة ومفيدة.,مدن نروشتة صريعة و مفيدة.,1.0,0.0,1.0,طيب إحنا كده عرفنا الأخطاء عشان ما نقعش فيها، ...,طيب إحنا كده عرفنا الأخطاء عشان ما نقعش فيها. ...
102,wavs/general_38_00044.wav,general,0.200000,كيف تبدو الطقس في مصر في الصيف؟,مش مستحيل,مش مستحيل,جو مصر يعتبر رسباه في الحالتين,1.0,0.0,1.0,اللي بيوصل تحت الصفر او الخليج بحرارتها اللي ب...,اللي بيوصل تحت الصفر أو الخليج بحررتها اللي بت...
1050,wavs/sales_37_00061.wav,sales,0.185185,ما هو التحذير الذي يجب أن يقدمه المستخدمون بعد...,CTA لخطوة معينة يخدوها.,CTA لخطوة معينة يخدوها.,يتواصله، يسجله، يحمله حاجة.,1.0,0.0,1.0,ركزي على الفايدة النهائية مش بس المواصفات. و ا...,ركزي على الفايدة النهائية مش بس المواصفات. و ا...
378,wavs/sales_17_00102.wav,sales,0.169811,ما هي التكاليف الإضافية التي ذكرتها؟,المصاريف الإضافية سمعت إنها أعلى غالباً؟ آه!,المصاريف الإضافية سمعت إنها أعلى غالباً؟ آه!,مصاريف صيانة ممكن تكون متخصصة أكتر,1.0,0.0,1.0,و ممكن المستأجر يضغط عشان يقلل الإجار لما يجي ...,و ممكن المستأجر يضغط عشان يقلل الإيجار لما ييج...
426,wavs/medical_26_00008.wav,medical,0.166667,أي نوع من الأدوية يُعرف باسم 'الفراولة القاتلة'؟,الترامادول,الترامادول,الترامدول,1.0,0.0,1.0,طيب خلينا نبدأ نفكك الأوهام دي التقرير بيكشف خ...,طيب خلينا نبدأ نفكك الأوهم دي التقرير بيكشف خر...
1078,wavs/sales_11_00020.wav,sales,0.151515,المعنى الأكبر من الجملة؟,العقد مش عشان نتخانق بيه ده عشان منوصلش أصلاً ...,العقد مش عشان نتخانق بيه ده عشان منوصلش أصلاً ...,قلية واضحة لحل النزاعات.,1.0,0.0,1.0,لازم يتضمن الية واضحة لحل النزاعات. و سيناريوه...,لازم يتضمن قلية واضحة لحل النزاعات. و سيناريوه...
564,wavs/sales_6_00092.wav,sales,0.150685,النسبة التي يجب أن تكون أقل من 35٪ لتجنب الأزم...,لو النسبة زادت عن 35%,لو النسبة زادت عن 35%,دي مخاطرة كبيرة جداً.,1.0,0.0,1.0,بالزبط. النسبة دي هي الحد الآمن اللي بيخلي الأ...,بالزبط. النسبة دي هي الحد الآمن اللي بيخلي الأ...


In [ ]:
import shutil

# =====
# ARTIFACT_DIR = "arat5_niletts_final_outputs"
# Path(ARTIFACT_DIR).mkdir(exist_ok=True)

# files_to_copy = [
#     "niletts_qa_test_reference_with_asr_context.csv",
#     "niletts_arat5_llm_reference_eval_predictions.csv",
#     "niletts_arat5_llm_asr_eval_predictions.csv",
#     "niletts_arat5_reference_vs_asr_comparison.csv",
#     "niletts_arat5_reference_vs_asr_summary_metrics.json",
#     "niletts_arat5_domain_level_results.csv",
#     "niletts_arat5_worst_asr_error_examples.csv",
# ]

# for f in files_to_copy:
#     if Path(f).exists():
#         shutil.copy(f, Path(ARTIFACT_DIR) / f)

# if Path(BEST_MODEL_DIR).exists():
#     if Path(ARTIFACT_DIR, BEST_MODEL_DIR).exists():
#         shutil.rmtree(Path(ARTIFACT_DIR, BEST_MODEL_DIR))
#     shutil.copytree(BEST_MODEL_DIR, Path(ARTIFACT_DIR, BEST_MODEL_DIR))

# zip_path = shutil.make_archive(ARTIFACT_DIR, "zip", ARTIFACT_DIR)
zip_path = shutil.make_archive(base_name=str(ARTIFACT_DIR), format="zip", root_dir=str(ARTIFACT_DIR))
# =====

print("Created:", zip_path)

Created: /root/araelectra_niletts_final_outputs.zip
